# 14.6 — Prophet

Prophet forecasts with an additive story: flexible trend, smooth seasonality, and sparse holiday effects.

Prophet packages decomposition, Fourier seasonality, and regularized changepoints into one practical baseline. This notebook implements the real additive ingredients directly in NumPy instead of installing Prophet. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 1306
rng = np.random.default_rng(SEED)


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def interval_coverage(y_true, lower, upper):
    y_true = np.asarray(y_true, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    inside = (y_true >= lower) & (y_true <= upper)
    return float(np.mean(inside))


def f13_time_series_ladder():
    t = np.arange(72, dtype=float)
    rungs = []

    y1 = np.full_like(t, 10.0)
    rungs.append({"name": "D1 constant", "t": t, "y": y1, "true": y1, "period": 12})

    y2 = 8.0 + 0.18 * t
    rungs.append({"name": "D2 drift", "t": t, "y": y2, "true": y2, "period": 12})

    noise3 = rng.normal(0.0, 0.45, size=t.size)
    true3 = 8.0 + 0.18 * t
    y3 = true3 + noise3
    rungs.append({"name": "D3 drift + noise", "t": t, "y": y3, "true": true3, "period": 12})

    seasonal4 = 1.8 * np.sin(2.0 * np.pi * t / 12.0)
    true4 = 8.0 + 0.12 * t + seasonal4
    y4 = true4 + rng.normal(0.0, 0.35, size=t.size)
    rungs.append({"name": "D4 seasonal", "t": t, "y": y4, "true": true4, "period": 12})

    seasonal5 = 1.8 * np.sin(2.0 * np.pi * t / 12.0)
    true5 = 8.0 + 0.10 * t + seasonal5
    true5 = true5 + np.where(t >= 48.0, 3.0 + 0.06 * (t - 48.0), 0.0)
    y5 = true5 + rng.normal(0.0, 0.45, size=t.size)
    y5[18] = y5[18] + 5.0
    y5[55] = y5[55] - 4.0
    rungs.append({"name": "D5 outliers + regime shift", "t": t, "y": y5, "true": true5, "period": 12})

    return rungs


def train_test_split_series(rung, train_size=48):
    train = {"t": rung["t"][:train_size], "y": rung["y"][:train_size]}
    test = {"t": rung["t"][train_size:], "y": rung["y"][train_size:], "true": rung["true"][train_size:]}
    return train, test


def print_ladder_preview(rungs):
    for rung in rungs:
        name = rung["name"]
        y = rung["y"]
        sample = np.round(y[:6], 3)
        print(f"{name}: n={y.size}, period={rung['period']}, first six={sample}")


def plot_forecast_panels(results, metric_name="RMSE", marker_key=None, component_key=None):
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    axes = axes.ravel()

    for idx, result in enumerate(results):
        ax = axes[idx]
        ax.plot(result["test_t"], result["test_y"], label="observed", color="black")
        ax.plot(result["test_t"], result["forecast"], label="forecast", color="tab:blue")
        ax.plot(result["test_t"], result["test_true"], label="true signal", color="tab:green", alpha=0.7)
        if component_key is not None and component_key in result:
            ax.plot(result["test_t"], result[component_key], label=component_key, color="tab:orange", alpha=0.8)
        if marker_key is not None and marker_key in result:
            marks = result[marker_key]
            for mark in np.atleast_1d(marks):
                ax.axvline(mark, color="tab:red", linestyle="--", alpha=0.6)
        ax.set_title(f"{result['name']}\n{metric_name}={result['metric']:.3f}")
        ax.grid(True, alpha=0.25)

    axes[-1].plot(range(1, len(results) + 1), [r["metric"] for r in results], marker="o")
    axes[-1].set_xticks(range(1, len(results) + 1))
    axes[-1].set_xlabel("D-rung")
    axes[-1].set_ylabel(metric_name)
    axes[-1].set_title(f"{metric_name} curve")
    axes[-1].grid(True, alpha=0.25)
    axes[0].legend(loc="best", fontsize=8)
    fig.tight_layout()



## The concept, built once (D1)

Prophet's core story is additive:

$$y(t)=g(t)+s(t)+h(t)+\varepsilon_t$$

We implement a NumPy version with piecewise-linear trend, Fourier seasonality, event indicators, and ridge-regularized changepoint deltas. The lesson numbers must hold exactly: $g(8)=13.400$, the weekly Fourier value at $t=3$ is $-0.033201$, and the event forecast at $t=6$ is $13.018169$.


In [ ]:

def prophet_like_additive(t, base_slope, intercept, changepoints, deltas, period, season_coefs, events):
    t = np.asarray(t, dtype=float)
    trend = intercept + base_slope * t

    for changepoint, delta in zip(changepoints, deltas):
        trend = trend + delta * np.maximum(0.0, t - changepoint)

    season = np.zeros_like(t, dtype=float)
    for order, sin_coef, cos_coef in season_coefs:
        angle = 2.0 * np.pi * order * t / period
        season = season + sin_coef * np.sin(angle)
        season = season + cos_coef * np.cos(angle)

    holiday = np.zeros_like(t, dtype=float)
    for event_time, lift in events.items():
        holiday = holiday + lift * (t == float(event_time))

    forecast = trend + season + holiday
    return {"trend": trend, "season": season, "holiday": holiday, "forecast": forecast}

lesson_trend = prophet_like_additive(
    np.array([8.0]),
    base_slope=0.5,
    intercept=10.0,
    changepoints=[6.0],
    deltas=[-0.3],
    period=7.0,
    season_coefs=[],
    events={},
)
lesson_fourier = prophet_like_additive(
    np.array([3.0]),
    base_slope=0.0,
    intercept=0.0,
    changepoints=[],
    deltas=[],
    period=7.0,
    season_coefs=[(1, 2.0, 1.0)],
    events={},
)
lesson_event = prophet_like_additive(
    np.array([6.0]),
    base_slope=0.3,
    intercept=10.0,
    changepoints=[],
    deltas=[],
    period=7.0,
    season_coefs=[(1, 1.0, 0.0)],
    events={6.0: 2.0},
)

assert np.isclose(lesson_trend["trend"][0], 13.400, atol=1e-12)
assert np.isclose(lesson_fourier["season"][0], -0.033201, atol=5e-7)
assert np.isclose(lesson_event["forecast"][0], 13.018169, atol=5e-7)
print("lesson checks passed")



Now fit the same structure. Candidate changepoints are hinge features $(t-s)_+$, Fourier terms encode smooth seasonality, and a ridge penalty shrinks changepoint deltas so trend bends do not freely memorize noise.


In [ ]:

def fit_prophet_like(train_t, train_y, period=12, order=2, changepoints=None, events=None, ridge_delta=20.0):
    train_t = np.asarray(train_t, dtype=float)
    train_y = np.asarray(train_y, dtype=float)

    if changepoints is None:
        changepoints = np.quantile(train_t, [0.35, 0.55, 0.75])

    if events is None:
        events = {}

    columns = [np.ones_like(train_t), train_t]
    penalties = [0.0, 0.0]

    for changepoint in changepoints:
        columns.append(np.maximum(0.0, train_t - changepoint))
        penalties.append(ridge_delta)

    for k in range(1, order + 1):
        angle = 2.0 * np.pi * k * train_t / period
        columns.append(np.sin(angle))
        columns.append(np.cos(angle))
        penalties.append(0.1)
        penalties.append(0.1)

    event_times = list(events.keys())
    for event_time in event_times:
        columns.append((train_t == float(event_time)).astype(float))
        penalties.append(1.0)

    X = np.column_stack(columns)
    penalty = np.diag(penalties)
    beta = np.linalg.solve(X.T @ X + penalty, X.T @ train_y)

    def predict(new_t, include_season=True):
        new_t = np.asarray(new_t, dtype=float)
        cols = [np.ones_like(new_t), new_t]
        for changepoint in changepoints:
            cols.append(np.maximum(0.0, new_t - changepoint))
        for k in range(1, order + 1):
            angle = 2.0 * np.pi * k * new_t / period
            if include_season:
                cols.append(np.sin(angle))
                cols.append(np.cos(angle))
            else:
                cols.append(np.zeros_like(new_t))
                cols.append(np.zeros_like(new_t))
        for event_time in event_times:
            cols.append((new_t == float(event_time)).astype(float))
        X_new = np.column_stack(cols)
        return X_new @ beta

    return {"beta": beta, "changepoints": changepoints, "predict": predict}



## The dataset ladder

We use the same F13 ladder in every notebook so the method faces increasing time-series difficulty inline: D1 constant, D2 drift, D3 drift plus noise, D4 monthly seasonality, and D5 outliers plus a real regime shift.


In [ ]:

rungs = f13_time_series_ladder()
print_ladder_preview(rungs)


In [ ]:

results = []

for rung in rungs:
    train, test = train_test_split_series(rung)
    events = {18.0: 0.0}
    model = fit_prophet_like(train["t"], train["y"], period=rung["period"], events=events)
    forecast = model["predict"](test["t"])
    metric = rmse(test["true"], forecast)
    results.append({
        "name": rung["name"],
        "test_t": test["t"],
        "test_y": test["y"],
        "test_true": test["true"],
        "forecast": forecast,
        "metric": metric,
        "changepoints": model["changepoints"],
    })

for result in results:
    print(f"{result['name']}: RMSE={result['metric']:.3f}")


In [ ]:

plot_forecast_panels(results, metric_name="RMSE", marker_key="changepoints")



## Pitfall on D5: letting changepoints explain seasonality

If we omit Fourier terms, hinge features can bend to chase repeated seasonal peaks and troughs. The fix is to keep explicit seasonality and shrink changepoint deltas.


In [ ]:

d5 = rungs[-1]
train, test = train_test_split_series(d5)
wrong = fit_prophet_like(train["t"], train["y"], period=d5["period"], order=0, ridge_delta=0.2)
fixed = fit_prophet_like(train["t"], train["y"], period=d5["period"], order=2, ridge_delta=20.0)
wrong_forecast = wrong["predict"](test["t"])
fixed_forecast = fixed["predict"](test["t"])
wrong_rmse = rmse(test["true"], wrong_forecast)
fixed_rmse = rmse(test["true"], fixed_forecast)
print(f"no Fourier, loose changepoints RMSE={wrong_rmse:.3f}")
print(f"Fourier + shrinkage RMSE={fixed_rmse:.3f}")
print(f"improvement={wrong_rmse - fixed_rmse:.3f}")



## Evaluate it + Practice

- Compare rolling-origin RMSE against a seasonal-naive baseline; Prophet-like structure should win when both trend and seasonality matter.
- Sanity check: with no changepoints and no events, the model should reduce to linear trend plus Fourier seasonality.
- Ablation: remove Fourier terms; D4 and D5 should worsen because trend bends try to mimic cycles.
- Failure signal: many large slope deltas clustered every season means changepoints are explaining seasonality.
- Track interval coverage if residual bands are added; narrow bands are not useful when coverage collapses.

Practice prompts:
1. Increase the changepoint penalty and inspect which D5 slope changes remain.


2. Modify the hardest rung and rerun the same metric table.

3. Write one sentence explaining whether RMSE or coverage is the better decision metric here.